# Capstone Report Notebook Shell

Use this notebook as the **narrative notebook** for your capstone.

This notebook is **not** the main engine. The main engine is:

- `run_pipeline.py`
- `dashboard_app.py`

Recommended workflow:

1. update `fund_manager_control.xlsx`
2. run `python run_pipeline.py`
3. launch `streamlit run dashboard_app.py`
4. use this notebook to organize analysis, interpretation, and report-ready tables/figures


## 0. Setup and Project Paths

This cell sets project paths and loads the most common output tables.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
TABLES_DIR = PROJECT_ROOT / "outputs" / "tables"
FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TABLES_DIR exists:", TABLES_DIR.exists())
print("FIGURES_DIR exists:", FIGURES_DIR.exists())

def safe_read_csv(path: Path) -> pd.DataFrame:
    if not path.exists():
        print(f"[missing] {path.name}")
        return pd.DataFrame()
    try:
        return pd.read_csv(path)
    except Exception as exc:
        print(f"[error] {path.name}: {exc}")
        return pd.DataFrame()

tables = {
    "inputs": safe_read_csv(TABLES_DIR / "tbl_inputs.csv"),
    "constraints": safe_read_csv(TABLES_DIR / "tbl_constraints.csv"),
    "inherited": safe_read_csv(TABLES_DIR / "tbl_inherited_fund.csv"),
    "candidates": safe_read_csv(TABLES_DIR / "tbl_candidates.csv"),
    "portfolio": safe_read_csv(TABLES_DIR / "tbl_portfolio_summary.csv"),
    "performance": safe_read_csv(TABLES_DIR / "tbl_performance_summary.csv"),
    "legacy_daily": safe_read_csv(TABLES_DIR / "tbl_legacy_fund_daily.csv"),
    "candidate_screen": safe_read_csv(TABLES_DIR / "tbl_candidate_screen.csv"),
    "static_compare": safe_read_csv(TABLES_DIR / "tbl_backtest_legacy_static_benchmark.csv"),
    "active_compare": safe_read_csv(TABLES_DIR / "tbl_backtest_legacy_static_active_benchmark.csv"),
    "factor_capm": safe_read_csv(TABLES_DIR / "tbl_factor_capm_summary.csv"),
    "factor_rolling_beta": safe_read_csv(TABLES_DIR / "tbl_factor_rolling_beta.csv"),
    "scenario_mc_summary": safe_read_csv(TABLES_DIR / "tbl_scenario_monte_carlo_summary.csv"),
    "scenario_stress": safe_read_csv(TABLES_DIR / "tbl_scenario_stress_summary.csv"),
    "manifest": safe_read_csv(TABLES_DIR / "tbl_project_manifest.csv"),
}
list(tables.keys())


## 1. Project Setup and Workbook Inputs

Use this section to summarize:
- project title
- team members
- benchmark
- rebalance settings
- active rule
- key assumptions


In [ ]:

tables["inputs"]


In [ ]:

tables["constraints"]


### Notes for the report
- What is the project title?
- What benchmark are you using?
- What active rule are you using?
- What constraints matter most for implementation?


## 2. Inherited Fund Audit

Required goals:
- reconstruct the inherited fund
- explain concentration
- show historical performance and drawdown
- identify weaknesses before redesign


In [ ]:
from IPython.display import Image, display
import subprocess, sys

# Auto-generate audit outputs if not already present
audit_csv = TABLES_DIR / "tbl_audit_stock_stats.csv"
if not audit_csv.exists():
    print("Running inherited_fund_analysis.py to generate audit outputs...")
    result = subprocess.run(
        [sys.executable, str(PROJECT_ROOT / "inherited_fund_analysis.py")],
        capture_output=True, text=True,
    )
    print(result.stdout[-2000:] if result.stdout else "")
    if result.returncode != 0:
        print("ERROR:", result.stderr[-1000:])

audit_stats = safe_read_csv(audit_csv)
weights_snapshot = safe_read_csv(TABLES_DIR / "tbl_legacy_weights_snapshot.csv")
print(f"Audit stats loaded: {audit_stats.shape[0]} stocks")

### Holdings at Inception (2010-01-01)

The fund was seeded with $1,000,000 equally split across 10 stocks.

In [ ]:
tables["inherited"][["legacy_ticker", "company_name", "sector", "initial_weight_2010"]]

### Per-Stock Statistics (In-Sample: 2010–2019)

All metrics use only information available through 2019-12-31.

| Column | Meaning |
|---|---|
| `total_return` / `ann_return` | Cumulative and annualized return over the full in-sample window |
| `ann_vol` | Annualized daily return volatility |
| `sharpe` / `sortino` | Risk-adjusted return (Sharpe = return/vol; Sortino = return/downside vol) |
| `max_drawdown` | Worst peak-to-trough loss |
| `alpha_ann` | Annualized CAPM alpha vs SPY — value added beyond passive exposure |
| `beta` | Market sensitivity (>1 = amplifies SPY moves) |
| `corr_to_spy` / `avg_pairwise_corr` | Correlation to benchmark and average correlation to the other 9 holdings |
| `return_1y` / `return_3y` | Return over the final 1 and 3 years before the decision date |
| `weight_start` / `weight_end` | Initial 10% equal weight vs drifted weight at 2019-12-31 |

In [ ]:
fmt = audit_stats.copy()
for col in ["total_return","ann_return","ann_vol","max_drawdown","return_1y","return_3y","weight_start","weight_end"]:
    if col in fmt.columns:
        fmt[col] = fmt[col].map(lambda v: f"{v:.1%}" if pd.notna(v) else "—")
for col in ["sharpe","sortino","beta","r_squared","corr_to_spy","avg_pairwise_corr"]:
    if col in fmt.columns:
        fmt[col] = fmt[col].map(lambda v: f"{v:.2f}" if pd.notna(v) else "—")
if "alpha_ann" in fmt.columns:
    fmt["alpha_ann"] = fmt["alpha_ann"].map(lambda v: f"{v:.2%}" if pd.notna(v) else "—")
fmt

### Individual Stock Returns vs SPY (2010–2019)

Each subplot shows cumulative growth of $1 for that stock (blue) vs SPY (gray dashed).

In [ ]:
display(Image(str(FIGURES_DIR / "fig_audit_grid_returns.png"), width=1100))

### Risk vs Return (2010–2019)

Each stock plotted by annualized volatility (x) and annualized return (y). SPY is the orange star benchmark reference.

In [ ]:
display(Image(str(FIGURES_DIR / "fig_audit_risk_return.png"), width=750))

### Weight Drift — Inception to Decision Date (2019-12-31)

Equal-weight start (10% each) vs how each position drifted by the manager handover date. Stocks that outperformed grew; underperformers shrank.

In [ ]:
display(Image(str(FIGURES_DIR / "fig_audit_weight_drift.png"), width=900))

### Pairwise Correlation Heatmap (2010–2019)

Red = highly correlated pairs (less diversification benefit). Green = lower correlation (more independent movement).

In [ ]:
display(Image(str(FIGURES_DIR / "fig_audit_correlation_heatmap.png"), width=750))

### Overall Legacy Fund Performance (Full Period)

In [ ]:
tables["performance"]

### Interpretation prompts
- Which inherited names became dominant by January 2020?
- How concentrated did the legacy fund become?
- What are the biggest risks visible in the inherited fund?
- Why might a redesign be justified?


## 3. Candidate Stock Analysis

Required goals:
- research 5–10 candidates
- show screening logic
- justify inclusion or rejection


In [ ]:

tables["candidates"]


In [ ]:

tables["candidate_screen"].sort_values(by=[c for c in ["sharpe_pre2020", "ann_return_pre2020"] if c in tables["candidate_screen"].columns], ascending=False).head(10)


### Interpretation prompts
- What is your candidate universe?
- Why do these names fit your theme or diversification goal?
- Which candidates look strongest by the pre-2020 screen?
- Which names did you reject, and why?


## 4. Revised Fund Design

Required goals:
- explain which original names were kept or dropped
- explain which new names were added
- explain target weights


In [ ]:

tables["portfolio"]


### Interpretation prompts
- Which inherited holdings did you keep?
- Which inherited holdings did you drop?
- Which new stocks entered the final 10-stock fund?
- Are the weights intuitive, or are they too concentrated?


## 5. Backtest and Performance Evaluation

Required comparison:
- Legacy Fund
- Revised Static Fund
- Revised Active Fund
- benchmark


In [ ]:

tables["static_compare"].head()


In [ ]:

tables["active_compare"].head()


In [ ]:

tables["performance"]


### Interpretation prompts
- Did the static redesign improve the inherited fund?
- Did active management improve the static redesign?
- How large are the tradeoffs in return, risk, drawdown, and turnover?
- Are any results too good to trust without caution?


## 6. Factor / Regression Analysis

Required goal:
- explain portfolio behavior with at least one factor/regression-based section


In [ ]:

tables["factor_capm"]


In [ ]:

tables["factor_rolling_beta"].head()


### Interpretation prompts
- What do alpha and beta suggest for each portfolio?
- How stable or unstable is beta over time?
- Does the active fund materially change market exposure?


## 7. Scenario / Stress Analysis

Required goal:
- show at least one simulation or stress-testing block


In [ ]:

tables["scenario_mc_summary"]


In [ ]:

tables["scenario_stress"]


### Interpretation prompts
- Which portfolio looks most resilient under simulated stress?
- Which scenario hurts the revised fund most?
- Does the redesign improve resilience relative to the legacy fund?


## 8. Final Recommendation

Write your manager recommendation here.

You should answer:
- Should the inherited fund be changed?
- What is your revised recommendation?
- Why is it better?
- What risks remain?
- Under what conditions might it fail?


## 9. Limitations

Be explicit and honest.
- data limitations
- model limitations
- sensitivity to assumptions
- implementation realism
- possible overfitting


## 10. Appendix

Use this section for:
- extra tables
- extra charts
- robustness checks
- parameter sensitivity
- additional notes


In [ ]:

tables["manifest"].tail()
